# Lab 5: Training a Simple Perceptron

## What this notebook does

Defines `train(model, X, y, epochs, learning_rate)`, implementing the perceptron learning rule: for each misclassified example, `weights += learning_rate * target * x` and `bias += learning_rate * target`. Runs it for several epochs over a toy dataset, then over real review data.

The full explanation of the learning rule, looping over epochs, and what "error" means lives in the reading pane above.

## Setup: re-including lab 4's `Perceptron`

This notebook is self-contained, so instead of importing `Perceptron` from a lab 4 file (which is what `course/lab_05_training_perceptron/train.py` does on disk, via `sys.path` -- a real pattern worth seeing at least once outside a notebook), we just redefine the same class here.

In [ ]:
%matplotlib inline
import re

import matplotlib.pyplot as plt
import numpy as np


class Perceptron:
    def __init__(self, n_features: int):
        self.weights = np.zeros(n_features)
        self.bias = 0.0

    def net_input(self, x: np.ndarray) -> float:
        return float(np.dot(x, self.weights) + self.bias)

    def activation(self, z: float) -> int:
        return 1 if z >= 0 else -1

    def predict(self, x: np.ndarray) -> int:
        return self.activation(self.net_input(x))

### Concept 1 practice: looping over epochs and data

Same nested for-loop shape as the training loop, this time counting sixes across rounds of dice rolls.

In [ ]:
rounds = [[3, 5, 2], [6, 6, 1], [4, 4, 4]]
total_sixes = 0
for round_rolls in rounds:
    for roll in round_rolls:
        if roll == 6:
            total_sixes += 1
print(total_sixes)  # 2


### Concept 2 practice: the perceptron learning rule

Same direction-times-magnitude-times-learning-rate update, this time adjusting a coffee brew time.

In [ ]:
brew_minutes = 5.0  # minutes used in the too-bitter batch
too_bitter = -1     # -1 means "too bitter", +1 would mean "too weak"
learning_rate = 0.2

adjustment = learning_rate * too_bitter * brew_minutes
brew_minutes += adjustment
print(brew_minutes)  # 4.0 - a little less brew time next time


### Concept 3 practice: error, loss, and mutating state

Same in-place `+=` accumulation as the bank balance example, this time a running game score.

In [ ]:
score = 0
score += 10  # mutates score in place
score += 25
print(score)  # 35 - remembers every point scored before it


### Concept 4 practice: plotting with matplotlib

Same `plt.subplots(1, 2, ...)` side-by-side layout, this time steps and drink orders.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].plot([1, 2, 3], [10, 20, 15])
axes[0].set_title("daily steps")
axes[1].bar(["tea", "coffee"], [5, 9])
axes[1].set_title("drink orders")
plt.tight_layout()
plt.savefig("two_panels.png")
print("Saved side-by-side charts to two_panels.png")  # Saved side-by-side charts to two_panels.png


## Exercise

Implement the perceptron learning rule as `train(model, X, y, epochs, learning_rate)`. The asserts check it converges to zero errors on a tiny, perfectly linearly separable toy set.

In [ ]:
def train(model, X, y, epochs: int = 20, learning_rate: float = 0.1) -> list[int]:
    """Run the perceptron learning rule. Returns mistakes made per epoch."""
    # TODO: for each epoch, loop over (xi, target) pairs. If model.predict(xi)
    # != target, update model.weights and model.bias using the rule above and
    # count it as a mistake. Return the list of per-epoch mistake counts.
    raise NotImplementedError


X_toy = np.array([[2.0, 2.0], [3.0, 3.0], [-2.0, -2.0], [-3.0, -1.0]])
y_toy = np.array([1, 1, -1, -1])

model = Perceptron(n_features=2)
errors = train(model, X_toy, y_toy, epochs=10, learning_rate=0.1)

assert errors[-1] == 0, "should converge to zero training errors on a linearly separable set"
for xi, target in zip(X_toy, y_toy):
    assert model.predict(xi) == target

print("Training converged. Errors per epoch:", errors)

## Solution

In [ ]:
def train(model, X, y, epochs: int = 20, learning_rate: float = 0.1) -> list[int]:
    errors_per_epoch = []
    for _ in range(epochs):
        errors = 0
        for xi, target in zip(X, y):
            prediction = model.predict(xi)
            if prediction != target:
                update = learning_rate * target
                model.weights += update * xi
                model.bias += update
                errors += 1
        errors_per_epoch.append(errors)
    return errors_per_epoch


model = Perceptron(n_features=2)
print(train(model, X_toy, y_toy, epochs=10, learning_rate=0.1))

## Real-world project

Reuses lab 4's `build_vocab`/`vectorize`/`pca_2d` pipeline on the same real review dataset, splits it 80/20 with `train_test_split`, trains the perceptron for 60 epochs with `train`, and reports `accuracy` on the held-out test set -- typically in the 70s-percent range, well above the untrained 50% baseline from lab 4. Real, messy, human-written text is noisy -- a linear model won't get everything right, and that's an honest result, not a bug.

In [ ]:
"""Load the shared real-world review dataset (embedded so this notebook needs no repo checkout)."""
import csv
import io

REVIEW_CSV = """sentence,label
the movie was absolutely wonderful and kept me hooked the whole time,1
i loved every minute of this film,1
the acting was superb and the story was touching,1
this was one of the best movies i have seen all year,1
the plot was engaging and the pacing was perfect,1
a brilliant film with outstanding performances,1
the soundtrack was beautiful and matched the mood perfectly,1
i would watch this movie again without hesitation,1
the director did an amazing job telling this story,1
this film exceeded all of my expectations,1
the characters felt real and the dialogue was sharp,1
a truly moving and unforgettable experience,1
the visuals were stunning and the effects were impressive,1
this is a masterpiece that everyone should see,1
the humor was clever and the timing was excellent,1
i was on the edge of my seat the entire film,1
the ending was satisfying and well earned,1
this movie made me laugh and cry at the same time,1
a wonderfully crafted story with great performances,1
the chemistry between the leads was fantastic,1
i really enjoyed the twist near the end,1
this film is a delightful surprise from start to finish,1
the cinematography was gorgeous throughout the movie,1
a heartwarming tale that left me smiling,1
the pacing kept the tension high without ever dragging,1
i cannot recommend this movie enough,1
the product works exactly as advertised and arrived quickly,1
i am very happy with this purchase,1
the quality is excellent for the price,1
this item exceeded my expectations in every way,1
the customer service team was friendly and helpful,1
setup was quick and the instructions were clear,1
this gadget has made my daily routine so much easier,1
the build quality feels sturdy and well made,1
i would definitely buy this product again,1
the battery life is impressive and lasts all day,1
shipping was fast and the packaging was excellent,1
this is by far the best purchase i made this year,1
the design is sleek and it works flawlessly,1
i appreciate how easy this product is to use,1
the price is fair given the quality you get,1
this laptop is fast reliable and great for work,1
the headphones sound amazing and are comfortable to wear,1
i love how lightweight and portable this device is,1
the app is intuitive and runs smoothly on my phone,1
this coffee maker brews a delicious cup every morning,1
the movie was boring and dragged on for way too long,0
i did not enjoy this film at all,0
the acting felt wooden and the story made no sense,0
this was one of the worst movies i have seen all year,0
the plot was confusing and the pacing was terrible,0
a disappointing film with weak performances,0
the soundtrack was forgettable and did not fit the mood,0
i would never watch this movie again,0
the director seemed to have no clear vision for this story,0
this film fell far short of my expectations,0
the characters felt flat and the dialogue was awkward,0
a frustrating and forgettable experience overall,0
the visuals were dull and the effects looked cheap,0
this movie is a mess that nobody should waste time on,0
the humor fell flat and the timing was off,0
i kept checking the clock throughout the entire film,0
the ending felt rushed and unearned,0
this movie made me want to leave the theater early,0
a poorly crafted story with unconvincing performances,0
the chemistry between the leads felt forced,0
i was annoyed by the predictable twist near the end,0
this film is a tedious slog from start to finish,0
the cinematography was flat and uninspired throughout,0
a dull tale that left me checking my phone,0
the pacing dragged and the tension never built,0
i cannot recommend this movie to anyone,0
the product broke after only two days of use,0
i am very unhappy with this purchase,0
the quality is poor for the price charged,0
this item fell short of my expectations in every way,0
the customer service team was rude and unhelpful,0
setup was confusing and the instructions were unclear,0
this gadget has made my daily routine more frustrating,0
the build quality feels flimsy and cheaply made,0
i would never buy this product again,0
the battery life is disappointing and drains quickly,0
shipping was slow and the packaging was damaged,0
this is by far the worst purchase i made this year,0
the design is clunky and it barely works,0
i am frustrated by how hard this product is to use,0
the price is too high given the poor quality,0
this laptop is slow unreliable and bad for work,0
the headphones sound tinny and are uncomfortable to wear,0
i hate how heavy and bulky this device is,0
the app is confusing and crashes on my phone,0
this coffee maker leaks and ruins my morning,0
the sequel was even better than the original film,1
i found the story dull and the characters uninteresting,0
the special effects were groundbreaking and beautifully done,1
the remote control stopped working within a week,0
this phone case fits perfectly and looks great,1
the instructions were missing several important steps,0
the keyboard feels responsive and is a joy to type on,1
the screen arrived scratched and the seller ignored my complaint,0
this backpack is durable and has plenty of storage,1
the movie relied on tired clichés and predictable jokes,0
the restaurant served delicious food with generous portions,1
our waiter was attentive and the service was outstanding,1
the pasta was cooked perfectly and full of flavor,1
this cafe has a cozy atmosphere and friendly staff,1
the steak was tender and cooked exactly how i asked,1
dessert was the highlight of an already fantastic meal,1
the wine pairing recommendations were spot on,1
i loved the fresh ingredients and creative menu,1
the bakery bread is soft and always baked fresh,1
this bistro has become my favorite spot in town,1
the book kept me turning pages late into the night,1
the author writes with wit and genuine emotional depth,1
i finished this novel in a single weekend it was that good,1
the world building in this book is rich and immersive,1
this memoir was honest funny and deeply moving,1
the mystery had twists i never saw coming,1
the poetry collection was beautiful and thought provoking,1
i recommend this book to anyone who loves a good story,1
the hotel room was spotless and beautifully decorated,1
the staff greeted us warmly and upgraded our room,1
the pool area was relaxing and well maintained,1
check in was smooth and the concierge was helpful,1
the breakfast buffet had a wonderful variety of fresh food,1
our stay at this resort exceeded every expectation,1
the view from the balcony was absolutely breathtaking,1
the gym and spa facilities were top notch,1
this hotel is perfect for a relaxing weekend getaway,1
the car handles smoothly and the ride is comfortable,1
fuel efficiency on this vehicle has been impressive,1
the dealership made the buying process quick and painless,1
this vacuum cleaner picks up dirt with ease,1
the blender is powerful and easy to clean,1
these shoes are comfortable and have held up well,1
the jacket is warm stylish and fits perfectly,1
the restaurant food was bland and arrived cold,0
our waiter seemed annoyed and service was painfully slow,0
the pasta was overcooked and lacked any real flavor,0
this cafe was noisy and the staff were dismissive,0
the steak was tough and clearly overdone,0
dessert was stale and ruined an otherwise average meal,0
the wine pairing recommendations were completely off,0
i disliked the stale ingredients and boring menu,0
the bakery bread is dry and often stale,0
this bistro has become a place i actively avoid,0
the book dragged and i never finished reading it,0
the author writes with clumsy prose and shallow characters,0
i abandoned this novel halfway through out of boredom,0
the world building in this book felt lazy and inconsistent,0
this memoir was dull repetitive and emotionally flat,0
the mystery was predictable from the very first page,0
the poetry collection felt uninspired and forgettable,0
i would not recommend this book to anyone,0
the hotel room was dirty and poorly maintained,0
the staff ignored us and refused to help with our request,0
the pool area was filthy and badly maintained,0
check in took forever and the concierge was unhelpful,0
the breakfast buffet had stale food and few options,0
our stay at this resort fell far below expectations,0
the view from the balcony was blocked by construction,0
the gym and spa facilities were outdated and dirty,0
this hotel is a poor choice for any kind of getaway,0
the car rattles constantly and the ride is uncomfortable,0
fuel efficiency on this vehicle has been disappointing,0
the dealership made the buying process stressful and confusing,0
this vacuum cleaner barely picks up any dirt,0
the blender is weak and difficult to clean,0
these shoes are uncomfortable and fell apart quickly,0
the jacket is thin flimsy and does not fit well,0
"""

def load_reviews():
    reader = csv.DictReader(io.StringIO(REVIEW_CSV))
    sentences, labels = [], []
    for row in reader:
        sentences.append(row["sentence"])
        labels.append(int(row["label"]))
    return sentences, labels

sentences, labels = load_reviews()
print(f"Loaded {len(sentences)} real review sentences.")


In [ ]:
MIN_WORD_COUNT = 3
MIN_POLARITY_RATIO = 0.7
TEST_FRACTION = 0.2
RANDOM_SEED = 0

y = np.array([1 if label == 1 else -1 for label in labels])


def tok(sentence: str) -> list[str]:
    return re.findall(r"[a-z]+", sentence.lower())


def build_vocab(sentences: list[str], y: np.ndarray) -> list[str]:
    pos_doc_counts: dict[str, int] = {}
    neg_doc_counts: dict[str, int] = {}
    for sentence, label in zip(sentences, y):
        bucket = pos_doc_counts if label == 1 else neg_doc_counts
        for word in set(tok(sentence)):
            bucket[word] = bucket.get(word, 0) + 1

    vocab = []
    for word in set(pos_doc_counts) | set(neg_doc_counts):
        pos, neg = pos_doc_counts.get(word, 0), neg_doc_counts.get(word, 0)
        total = pos + neg
        if total < MIN_WORD_COUNT:
            continue
        if abs(pos - neg) / total < MIN_POLARITY_RATIO:
            continue
        vocab.append((word, total))

    vocab.sort(key=lambda pair: pair[1], reverse=True)
    return [word for word, _ in vocab]


def vectorize(sentences: list[str], vocab: list[str]) -> np.ndarray:
    index = {word: i for i, word in enumerate(vocab)}
    X = np.zeros((len(sentences), len(vocab)))
    for row, s in enumerate(sentences):
        for w in set(tok(s)):
            if w in index:
                X[row, index[w]] = 1.0
    return X


def pca_2d(X: np.ndarray) -> np.ndarray:
    centered = X - X.mean(axis=0)
    cov = np.cov(centered, rowvar=False)
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    top2 = eigenvectors[:, np.argsort(eigenvalues)[::-1][:2]]
    return centered @ top2


def train_test_split(*arrays: np.ndarray, test_fraction: float, seed: int):
    rng = np.random.default_rng(seed)
    n = len(arrays[0])
    indices = rng.permutation(n)
    n_test = int(n * test_fraction)
    test_idx, train_idx = indices[:n_test], indices[n_test:]
    return [a[train_idx] for a in arrays], [a[test_idx] for a in arrays]


def accuracy(model, X: np.ndarray, y: np.ndarray) -> float:
    predictions = np.array([model.predict(x) for x in X])
    return float((predictions == y).mean())


vocab = build_vocab(sentences, y)
X = vectorize(sentences, vocab)
X2d = pca_2d(X)
print(f"Sentiment vocabulary ({len(vocab)} words): {vocab}")

(X_train, X2d_train, y_train), (X_test, X2d_test, y_test) = train_test_split(
    X, X2d, y, test_fraction=TEST_FRACTION, seed=RANDOM_SEED
)

model = Perceptron(n_features=X.shape[1])
errors_per_epoch = train(model, X_train, y_train, epochs=60, learning_rate=1.0)
accuracy_per_epoch = [1 - e / len(X_train) for e in errors_per_epoch]

test_acc = accuracy(model, X_test, y_test)
print(f"Train examples: {len(X_train)}, test examples: {len(X_test)}")
print(f"Held-out test accuracy: {test_acc:.2%}")

predictions_test = np.array([model.predict(x) for x in X_test])
correct = predictions_test == y_test

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(range(1, len(errors_per_epoch) + 1), accuracy_per_epoch, marker="o")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("training accuracy")
axes[0].set_title("Learning curve")

axes[1].scatter(*X2d_train[y_train == 1].T, color="seagreen", alpha=0.4, label="train +")
axes[1].scatter(*X2d_train[y_train == -1].T, color="crimson", alpha=0.4, label="train -")
axes[1].scatter(*X2d_test[correct].T, color="royalblue", marker="o", label="test correct")
axes[1].scatter(*X2d_test[~correct].T, color="black", marker="x", s=80, label="test wrong")
axes[1].set_xlabel("PCA component 1 (visualization only)")
axes[1].set_ylabel("PCA component 2 (visualization only)")
axes[1].set_title(f"Trained perceptron (test accuracy {test_acc:.0%})")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()